In [61]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from src.utils.load_config import load_config
from src.utils import common as cm
config = load_config()
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline


In [62]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

In [63]:
df = cm.read_dataframe(
    cm.resolve_path(config,"features_data"),
                    "properties_features.csv"
)

<project>/src/utils/common.py:22: DtypeWarning: Columns (0: parcel_id, 1: situs_city, 2: deed_book, 3: deed_page) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(os.path.join(path, filename), **kwargs)


In [64]:
df.columns

Index(['county', 'parcel_id', 'situs_street_num', 'situs_street_name',
       'situs_city', 'situs_zip', 'sale_price', 'sale_date', 'deed_book',
       'deed_page', 'land_value', 'building_value', 'total_value', 'acreage',
       'land_use_class', 'neighborhood_code', 'year_built', 'heated_area',
       'bedrooms', 'bathrooms', 'latitude', 'longitude', 'arms_length_sale',
       'property_age', 'price_per_sqft', 'neighborhood_median_price_per_sqft',
       'neighborhood_sale_count', 'access'],
      dtype='str')

In [65]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 829506 entries, 0 to 829505
Data columns (total 28 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   county                              829506 non-null  str    
 1   parcel_id                           829389 non-null  object 
 2   situs_street_num                    829455 non-null  float64
 3   situs_street_name                   829506 non-null  str    
 4   situs_city                          759175 non-null  str    
 5   situs_zip                           462188 non-null  float64
 6   sale_price                          635479 non-null  float64
 7   sale_date                           781433 non-null  str    
 8   deed_book                           828720 non-null  object 
 9   deed_page                           829051 non-null  object 
 10  land_value                          826929 non-null  float64
 11  building_value                      8

In [72]:
df.isna().groupby(df['county']).sum()


,county,parcel_id,situs_street_num,situs_street_name,situs_city,situs_zip,sale_price,sale_date,deed_book,deed_page,land_value,building_value,total_value,acreage,land_use_class,neighborhood_code,year_built,heated_area,bedrooms,bathrooms,latitude,longitude,arms_length_sale,property_age,price_per_sqft,neighborhood_median_price_per_sqft,neighborhood_sale_count,access
county,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Buncombe,0,0,0,0,69902,69902,69305,0,0,0,0,0,0,0,0,10316,11778,11778,11778,11778,0,0,0,11778,69569,48496,48496,0
Guilford,0,0,0,0,108,1927,38022,32964,5,5,2477,0,2477,0,0,654,11463,11648,11648,11648,0,0,0,11463,95325,1352,1352,0
Mecklenburg,0,117,51,0,0,295083,66639,83,437,450,100,100,100,0,0,99,102,100,683,12831,0,0,0,102,155233,256,256,0
Wake,0,0,0,0,321,406,20061,15026,344,0,0,0,0,0,0,0,1,0,311694,311694,311694,311694,0,1,146417,419,419,311694


In [46]:
df.isna().sum().sort_values(ascending=False)

price_per_sqft                        564824
bathrooms                             412407
bedrooms                              412227
access                                347112
latitude                              347112
longitude                             347112
sale_price                            269066
situs_city                            103408
property_age                           80415
year_built                             80415
sale_date                              73070
neighborhood_sale_count                67657
neighborhood_median_price_per_sqft     67657
heated_area                            48403
neighborhood_code                      12628
situs_street_num                        5083
land_value                              3118
total_value                             3118
deed_book                               1521
deed_page                                934
parcel_id                                817
land_use_class                           549
acreage   

In [47]:
def describe_property(row):
      parts = [f"{int(row.bedrooms)}-bed {int(row.bathrooms)}-bath {row.land_use_class} home"]
      if pd.notna(row.situs_city):
          parts.append(f"in {row.situs_city}, {row.county} County.")
      else:
          parts.append(f"in {row.county} County.")
      parts.append(f"{int(row.heated_area)} sqft")
      if pd.notna(row.year_built):
          parts.append(f"built {int(row.year_built)}")
      if pd.notna(row.acreage):
          parts.append(f"{row.acreage} acres")
      if pd.notna(row.total_value):
          parts.append(f"assessed at ${int(row.total_value):,}")
      return ", ".join(parts) + "."

print(describe_property(df.iloc[50000]))

4-bed 2-bath single_family home, in CHARLOTTE, Mecklenburg County., 1888 sqft, built 2017, 0.11233515 acres, assessed at $342,400.


In [48]:
df['situs_city'].unique()

<ArrowStringArray>
[      'CORNELIUS',        'MATTHEWS',       'CHARLOTTE',           'UNINC',
       'MINT HILL',    'HUNTERSVILLE',        'DAVIDSON',       'PINEVILLE',
       'STALLINGS',               nan,      'GREENSBORO',       'JAMESTOWN',
          'COLFAX',     'GIBSONVILLE',            'ELON',    'BROWN SUMMIT',
      'REIDSVILLE',       'RANDLEMAN',      'BURLINGTON',        'WHITSETT',
         'SEDALIA',         'LIBERTY',          'JULIAN',    'MCLEANSVILLE',
 'PLEASANT GARDEN',    'KERNERSVILLE',          'CLIMAX',     'SUMMERFIELD',
      'HIGH POINT',        'ARCHDALE',      'STOKESDALE',       'OAK RIDGE',
         'RALEIGH',         'WENDELL',            'CARY',     'WAKE FOREST',
      'KNIGHTDALE',   'FUQUAY VARINA',          'GARNER',            'APEX',
   'HOLLY SPRINGS',   'WILLOW SPRING',     'MORRISVILLE',         'ZEBULON',
      'ROLESVILLE',          'ANGIER',        'NEW HILL',     'YOUNGSVILLE',
          'DURHAM',         'CLAYTON',       'CREEDMOOR']

In [49]:
# df.isna().groupby(df["situs_city"]).sum()
df.isna().sum()

county                                     0
parcel_id                                817
situs_street_num                        5083
situs_street_name                          8
situs_city                            103408
sale_price                            269066
sale_date                              73070
deed_book                               1521
deed_page                                934
land_value                              3118
building_value                           131
total_value                             3118
acreage                                  255
land_use_class                           549
neighborhood_code                      12628
year_built                             80415
heated_area                            48403
bedrooms                              412227
bathrooms                             412407
latitude                              347112
longitude                             347112
arms_length_sale                           0
property_a

## Missingness by county

Missingness isn't random — each county's raw export has a different schema, so gaps are structural (MNAR), not sampling noise.

In [50]:
cols = ['bedrooms','bathrooms','longitude','latitude','area','situs_city',
        'price_momentum','education_nearest_miles','places_count','health_care_share','activity_share',
        'emergency_fire_miles','emergency_police_miles','emergency_medical_miles',
        'year_built','property_age','neighborhood_code','neighborhood_sale_count',
        'neighborhood_median_price_per_sqft','sale_price','price_per_sqft','heated_area','total_value']
df.groupby('county')[cols].apply(lambda g: g.isna().mean().round(3)).T

county,Buncombe,Guilford,Mecklenburg,Wake
bedrooms,0.288,0.113,0.051,1.000
bathrooms,0.288,0.113,0.052,1.000
longitude,0.000,0.000,0.000,1.000
latitude,0.000,0.000,0.000,1.000
access,0.000,0.000,0.000,1.000
situs_city,1.000,0.001,0.000,0.003
year_built,0.288,0.098,0.046,0.056
property_age,0.288,0.098,0.046,0.056
neighborhood_code,0.115,0.005,0.000,0.000
neighborhood_sale_count,0.597,0.021,0.004,0.006


Wake never captures bedrooms/bathrooms/coordinates; Buncombe never captures situs_city — these are hardcoded `pd.NA` in `data_preprocessing.py`, not missing-at-random gaps. `access` tracks `latitude`/`longitude` exactly since it's a distance-based feature computed from coordinates. `price_per_sqft` is gated on `arms_length_sale & heated_area > 0`, so it inherits missingness from both. (`owner_name`/`situs_zip` are no longer columns at all - dropped in `clean_properties()`.)

## Shape & categorical distributions

In [51]:
print(df['county'].value_counts())
print()
print(df['land_use_class'].value_counts(dropna=False))
print()
print(df['arms_length_sale'].value_counts(normalize=True))

county
Wake           347111
Mecklenburg    327623
Guilford       168189
Buncombe       102058
Name: count, dtype: int64

land_use_class
single_family    759078
other             79981
townhouse         49802
condo             31339
vacant_land       24232
NaN                 549
Name: count, dtype: int64

arms_length_sale
False    0.590396
True     0.409604
Name: proportion, dtype: float64


## Data quality checks

Only 38% of rows are `arms_length_sale` — most deed transfers are foreclosures, gifts, or bulk/portfolio sales, not real market signal. A few outlier checks below.

In [52]:
# duplicate parcels (same county + parcel_id appears more than once)
df.duplicated(subset=['county', 'parcel_id']).sum()

np.int64(3484)

In [53]:
# land_use_class mapping bug: rows tagged single_family that are clearly apartment/commercial buildings
df.loc[(df['land_use_class'] == 'single_family') & (df['bedrooms'] > 15)

      ].sort_values('bedrooms', ascending=False).head(10)

In [54]:
# sale_price outliers - all non-arms-length bulk/portfolio sales, so the arms_length filter
# is already protecting price_per_sqft from these
df.nlargest(10, 'sale_price')[['county', 'parcel_id', 'sale_price', 'land_use_class', 'arms_length_sale', 'sale_date']]

,county,parcel_id,sale_price,land_use_class,arms_length_sale,sale_date
602141,Wake,7050,2320500.0,single_family,True,2023-02-17
164535,Mecklenburg,166256.0,2320000.0,other,True,2018-04-27 00:00:00
604247,Wake,10420,2320000.0,single_family,True,2025-03-19
855822,Wake,378731,2320000.0,single_family,True,2025-03-31
308095,Mecklenburg,15509122.0,2319000.0,single_family,True,2022-09-22 00:00:00
748065,Wake,226611,2318000.0,single_family,False,2000-12-01
609384,Wake,18806,2316000.0,single_family,True,2023-07-11
635333,Wake,61681,2316000.0,single_family,True,2025-08-08
666110,Wake,111497,2315000.0,single_family,True,2025-11-20
317677,Mecklenburg,15503523.0,2313500.0,single_family,True,2021-01-04 00:00:00


In [55]:
# more of the single_family mislabeling - multi-million dollar / huge-sqft "single family" parcels
df.loc[df['land_use_class'] == 'single_family'].nlargest(10, 'heated_area')

In [56]:
# other rough edges: a negative building_value, and multi-thousand-acre "acreage" outliers
print('negative building_value rows:', (df['building_value'] < 0).sum())
df.nlargest(5, 'acreage')[['county', 'parcel_id', 'acreage', 'land_use_class', 'total_value']]

negative building_value rows: 0


,county,parcel_id,acreage,land_use_class,total_value
403610,Buncombe,71216776800000,12964.450195,other,6.482230e+07
660633,Wake,102676,4584.260000,other,1.212219e+09
631045,Wake,54411,1933.520000,other,3.463213e+07
361347,Buncombe,976410024500000,1923.550049,other,1.129550e+07
403167,Buncombe,978213265800000,1840.000000,other,5.140000e+06


## Signal check: what correlates with `total_value`?

`total_value` (assessed value) is the better target than `sale_price` — only ~1% missing vs. 69%+ for `sale_price`/`price_per_sqft`, and it's populated whether or not a sale ever happened. Restricting to a "sane" single-family subset (to avoid the mislabeled apartment complexes above):

In [57]:
sf = df[(df['land_use_class'] == 'single_family')
        & df['total_value'].between(10_000, 5_000_000)
        & df['heated_area'].between(300, 10_000)]
print('n rows (sane single_family):', len(sf))

feature_cols = ['total_value', 'heated_area', 'bedrooms', 'bathrooms', 'acreage',
                'property_age', 'neighborhood_median_price_per_sqft', 'price_momentum',
                'education_nearest_miles', 'places_count', 'health_care_share', 'activity_share',
                'emergency_fire_miles', 'emergency_police_miles', 'emergency_medical_miles']
sf[feature_cols].corr(numeric_only=True)['total_value'].sort_values(ascending=False)

n rows (sane single_family): 733795


total_value                           1.000000
heated_area                           0.730175
bathrooms                             0.626233
neighborhood_median_price_per_sqft    0.496687
bedrooms                              0.374905
access                                0.080591
acreage                               0.065795
property_age                         -0.223431
Name: total_value, dtype: float64

## Owner-name entity check: does an LLC/corp filter catch the mislabeled "single_family" outliers above?

`owner_name` is already dropped from `properties_features.csv` by the current
`data_preprocessing.py` pipeline, so this loads the raw pre-clean merge
(`properties_combined.csv`, untouched by cleaning) to check it directly.

In [ ]:
raw = cm.read_dataframe(cm.resolve_path(config, "processed_data"), "properties_combined.csv")
print("raw rows:", len(raw))

owner_upper = raw['owner_name'].fillna("").str.upper()
llc_or_corp = owner_upper.str.contains("LLC") | owner_upper.str.contains("CORP")
print(f"rows with LLC or CORP in owner_name: {llc_or_corp.sum()} ({llc_or_corp.mean():.2%})")
raw.loc[llc_or_corp, ['county', 'parcel_id', 'owner_name', 'total_value', 'land_use_class']].sample(10, random_state=42)

In [ ]:
# compare a bare LLC/CORP check against the broader keyword filter already in data_preprocessing.py
# (LLC/CORP alone misses TRUST, BANK, government bodies like "MECKLENBURG COUNTY", school boards, etc.)
from src.components.data_preprocessing import is_individual_owner

is_entity_broad = ~is_individual_owner(raw['owner_name'])
print(f"flagged by LLC/CORP only:             {llc_or_corp.sum():>7} ({llc_or_corp.mean():.2%})")
print(f"flagged by full keyword filter:       {is_entity_broad.sum():>7} ({is_entity_broad.mean():.2%})")
print(f"caught by full filter, missed by LLC/CORP alone: {(is_entity_broad & ~llc_or_corp).sum()}")

# owner names that still look like entities but neither check catches - remaining gaps
still_kept = raw.loc[~is_entity_broad & ~llc_or_corp, ['owner_name', 'total_value']].dropna()
still_kept.nlargest(15, 'total_value')

In [ ]:
# does the owner filter actually catch the extreme-total_value outliers that were
# inflating RMSE (the $103M/$71M/etc "single_family" rows from the training diagnosis)?
huge = raw.nlargest(20, 'total_value')[['county', 'parcel_id', 'owner_name', 'total_value', 'land_use_class']].copy()
huge['individual_owner'] = is_individual_owner(huge['owner_name'])
huge['llc_or_corp'] = huge['owner_name'].fillna('').str.upper().str.contains('LLC|CORP')
huge